In [1]:
import json
import re

INPUT_FILE = "who_structured_master_final_wtsd.json" # Your file with the manual fixes added
OUTPUT_FILE = "who_structured_master_cleaned_safe.json"

def fix_encoding(text):
    """Fixes common UTF-8 'Mojibake' corruption."""
    if not isinstance(text, str):
        return text
    replacements = {
        "â€”": "—", "â€™": "'", "â€œ": '"', "â€": '"', "â‚¹": "₹", "Â°": "°"
    }
    for corrupted, clean in replacements.items():
        text = text.replace(corrupted, clean)
    return text

def clean_mid_sentence_newlines(text):
    """Safely merges mid-sentence line breaks without destroying bullet points."""
    text = text.replace('\r\n', '\n')
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        stripped_line = line.strip()
        if not stripped_line:
            cleaned_lines.append("")
            continue
            
        if (cleaned_lines and cleaned_lines[-1] and 
            re.match(r'^[a-z]', stripped_line) and 
            not cleaned_lines[-1].strip().endswith(('.', '?', '!', ':', ';'))):
            cleaned_lines[-1] = f"{cleaned_lines[-1]} {stripped_line}"
        else:
            cleaned_lines.append(line)

    return '\n'.join(cleaned_lines)

def process_document(doc):
    for section in doc.get("content", []):
        section["section_title"] = fix_encoding(section.get("section_title", ""))
        raw_text = fix_encoding(section.get("text", ""))
        
        # Scrub References
        if "\nReferences\n" in raw_text:
            raw_text = raw_text.split("\nReferences\n")[0]
        elif "\nReferences" in raw_text:
            raw_text = raw_text.split("\nReferences")[0]
            
        # Clean line breaks
        section["text"] = clean_mid_sentence_newlines(raw_text)
            
    return doc

print("⏳ Loading master JSON...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    master_data = json.load(f)

cleaned_data = [process_document(doc) for doc in master_data]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print(f"✅ Success! Data safely cleaned to: {OUTPUT_FILE}")

⏳ Loading master JSON...
✅ Success! Data safely cleaned to: who_structured_master_cleaned_safe.json


In [2]:
import json

INPUT_FILE = "who_structured_master_cleaned_safe.json" # Use your current working file

print("⏳ Auditing Master JSON for missing sections...\n")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    master_data = json.load(f)

missing_key_facts = []
missing_overview = []
suspiciously_short = []

for doc in master_data:
    title = doc.get("title", "Unknown")
    sections = doc.get("content", [])
    
    # Extract all section titles and convert to lowercase for easy checking
    section_titles = [sec.get("section_title", "").lower() for sec in sections]
    
    # Calculate total text length of the document
    total_length = sum(len(sec.get("text", "")) for sec in sections)
    
    if "key facts" not in section_titles:
        missing_key_facts.append(title)
        
    if "overview" not in section_titles:
        missing_overview.append(title)
        
    # If the whole page has less than 500 characters, the scraper probably failed
    if total_length < 500:
        suspiciously_short.append(title)

print(f"🚨 Missing 'Key facts' ({len(missing_key_facts)} files):")
print(missing_key_facts)
print("-" * 50)

print(f"🚨 Missing 'Overview' ({len(missing_overview)} files):")
print(missing_overview)
print("-" * 50)

print(f"⚠️ Suspiciously Short Data / Possible Scrape Failures ({len(suspiciously_short)} files):")
print(suspiciously_short)

⏳ Auditing Master JSON for missing sections...

🚨 Missing 'Key facts' (19 files):
['Asthma', 'Cancer', 'Chagas disease (also known as American trypanosomiasis)', 'Family planning/contraception methods', 'Human rights', 'Influenza (seasonal)', 'Kidney disease', 'Lead poisoning', 'Malaria', 'Nipah virus', 'Noncommunicable diseases', 'Prequalification of health products', "Protecting workers' health", 'Respiratory syncytial virus (RSV)', 'Road traffic injuries', 'The top 10 causes of death', 'Universal health coverage (UHC)', 'White phosphorus', 'Yellow fever']
--------------------------------------------------
🚨 Missing 'Overview' (41 files):
['Abuse of older people', 'Blood safety and availability', 'Brucellosis', 'Buruli ulcer (Mycobacterium ulcerans infection)', 'Campylobacter', 'Cancer', 'Cardiovascular diseases (CVDs)', 'Depressive disorder (depression)', 'Drinking-water', 'Echinococcosis', 'Emergency contraception', 'Falls', 'Foodborne trematode infections', 'Hantavirus', 'Hepatiti

In [3]:
import json

# Look at your final, cleaned file
INPUT_FILE = "who_structured_master_cleaned_safe.json" 

print("⏳ Scanning for Tables in the Master JSON...\n")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    master_data = json.load(f)

tables_found = 0

for doc in master_data:
    title = doc.get("title", "Unknown")
    
    for section in doc.get("content", []):
        text = section.get("text", "")
        
        # A simple heuristic to find tables: multiple pipe characters "|"
        # or the presence of markdown table dividers like "|---"
        if text.count("|") > 4 or "|---" in text:
            tables_found += 1
            print(f"📌 Found a table in: {title} (Section: {section.get('section_title')})")
            print("-" * 50)
            
            # Print the first 300 characters of the table so you can verify it looks right
            print(text[:300] + "...\n")
            print("=" * 50)

print(f"✅ Scan complete! Found {tables_found} potential tables.")

⏳ Scanning for Tables in the Master JSON...

📌 Found a table in: Vector-borne diseases (Section: List of vector-borne diseases, according to their vector)
--------------------------------------------------
| Vector | Disease caused | Type of pathogen |
|---|---|---|
| Mosquito (Aedes) | Chikungunya | Virus |
| Mosquito (Aedes) | Dengue | Virus |
| Mosquito (Aedes) | Lymphatic filariasis | Parasite |
| Mosquito (Aedes) | Rift Valley fever | Virus |
| Mosquito (Aedes) | Yellow Fever | Virus |
| Mosquito...

✅ Scan complete! Found 1 potential tables.
